# TCGer iOS Scan Index — Parity Rebuild (Colab)

Rebuilds `CardsIndexVectors.bin` for the iOS scanner by re-embedding every catalog card image with the **same encoder the phone runs**: `facebook/dinov2-small` in fp32 (verified ≈0.999 cosine vs the CoreML fp16 conversion), using the exact iOS preprocessing (resize shortest edge 256 → center crop 224 → ImageNet norm). The shipped index was built with a q8-quantized ONNX model, which costs ~0.02–0.09 of similarity per card; this rebuild removes that gap.

**Inputs** (place in the Drive folder or upload to `/content/`):
1. Card images — already in Drive (`card-library/pokemon/...`, any layout; matched by filename).
2. `CardsIndexMetadata.json` — from the repo at `mobile-apps/ios/TCGer/TCGer/Resources/ScanIndex/` (defines row order).
3. `CardsIndexVectors.bin` (optional but recommended) — the current bin; rows whose image can't be found/fetched keep their old vector.

**Output**: `CardsIndexVectors-parity.bin` written to the Drive folder. Copy it into the repo as `ScanIndex/CardsIndexVectors.bin` and rebuild the app. The **web** index must stay q8-built (the browser runtime is q8) — this bin is iOS-only.

Progress is checkpointed to Drive, so a disconnected runtime resumes where it left off. Use a GPU runtime (Runtime → Change runtime type → T4).

In [ ]:
# ---- Config ----
DRIVE_FOLDER = "/content/drive/MyDrive/UniFi Drive_UNAS Pro 8/UNAS Pro 8_Main Backup/Images/pvc-19daba96-3902-4005-aab6-60b80b8f171a/card-library"
IMAGES_SUBDIR = "pokemon"                               # subfolder holding the images

# Metadata / old bin: first path that exists wins.
METADATA_CANDIDATES = [DRIVE_FOLDER + "/pokemon/CardsIndexMetadata.json", DRIVE_FOLDER + "/CardsIndexMetadata.json", "/content/CardsIndexMetadata.json"]
OLD_BIN_CANDIDATES  = [DRIVE_FOLDER + "/pokemon/CardsIndexVectors.bin", DRIVE_FOLDER + "/CardsIndexVectors.bin", "/content/CardsIndexVectors.bin"]

OUT_BIN   = DRIVE_FOLDER + "/CardsIndexVectors-parity.bin"
CHECKPOINT = DRIVE_FOLDER + "/parity-rebuild-checkpoint.npz"

COPY_IMAGES_LOCAL = True    # copy Drive images to the Colab disk first (much faster IO)
DOWNLOAD_MISSING = True     # fetch images missing from Drive from their imageURL
BATCH_SIZE = 64
SCALE = 127                 # int8 quantization scale — must match AnnoyIndexStore

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, struct, math
import numpy as np

meta_path = next((p for p in METADATA_CANDIDATES if os.path.exists(p)), None)
assert meta_path, f"CardsIndexMetadata.json not found in {METADATA_CANDIDATES} — copy it from the repo's ScanIndex folder"
meta = json.load(open(meta_path))
assert all(e["annIndex"] == i for i, e in enumerate(meta)), "annIndex order mismatch"
N = len(meta)
print(f"metadata: {N} cards from {meta_path}")

old_q = None
old_path = next((p for p in OLD_BIN_CANDIDATES if os.path.exists(p)), None)
if old_path:
    raw = open(old_path, "rb").read()
    cnt, dim = struct.unpack("<ii", raw[:8])
    assert cnt == N, f"old bin has {cnt} rows, metadata has {N}"
    old_q = np.frombuffer(raw, dtype=np.int8, offset=8, count=cnt*dim).reshape(cnt, dim).copy()
    print(f"old bin: {cnt} x {dim} from {old_path} (fallback rows available)")
else:
    dim = 384
    print("no old bin found — rows with missing images will be zero vectors (add CardsIndexVectors.bin to avoid this)")

In [ ]:
# ---- Locate images: recursive scan, match files to cardIds by common layouts ----
import pathlib, shutil, subprocess

src_root = os.path.join(DRIVE_FOLDER, IMAGES_SUBDIR)
assert os.path.isdir(src_root), f"images folder not found: {src_root}"

root = src_root
if COPY_IMAGES_LOCAL:
    root = "/content/card-images"
    if not os.path.isdir(root):
        print("copying images to local disk (one-time, can take a while on many small files)...")
        subprocess.run(["rsync", "-a", "--info=progress2", src_root + "/", root + "/"], check=True)
    print("using local copy:", root)

EXTS = {".png", ".jpg", ".jpeg", ".webp"}
files = [p for p in pathlib.Path(root).rglob("*") if p.suffix.lower() in EXTS]
print(f"found {len(files)} image files")

# Index files by candidate keys:
#   1. bare stem            e.g. sv03-136.webp            -> "sv03-136"
#   2. parent/stem          e.g. sv03/136.png             -> "sv03-136"
#   3. stem 'high'/'low' with card folder: sv03/136/high.webp -> "sv03-136"
by_key = {}
for p in files:
    stem = p.stem
    by_key.setdefault(stem, p)
    by_key.setdefault(f"{p.parent.name}-{stem}", p)
    if stem in ("high", "low"):
        by_key.setdefault(f"{p.parent.parent.name}-{p.parent.name}", p)

def find_image(card_id):
    for key in (card_id, card_id.replace("/", "_")):
        if key in by_key:
            return by_key[key]
    return None

have = sum(1 for e in meta if find_image(e["cardId"]) is not None)
print(f"matched {have}/{N} cards to Drive images ({N-have} missing{' — will download' if DOWNLOAD_MISSING else ''})")
missing_sample = [e["cardId"] for e in meta if find_image(e["cardId"]) is None][:10]
print("sample missing:", missing_sample)

In [ ]:
# ---- Model: facebook/dinov2-small fp32, CLS token, iOS-identical preprocessing ----
import torch
from transformers import Dinov2Model
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Dinov2Model.from_pretrained("facebook/dinov2-small").eval().to(device)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
print("device:", device)

def preprocess(img):
    """Mirror CardEmbeddingEncoder.swift: resize shortest edge 256 (bicubic,
    ceil), center crop 224. Returns HWC uint8 ndarray."""
    w, h = img.size
    s = max(256 / min(w, h), 224 / w, 224 / h)
    rw, rh = math.ceil(w * s), math.ceil(h * s)
    img = img.resize((rw, rh), Image.BICUBIC)
    cx, cy = max(0, (rw - 224) // 2), max(0, (rh - 224) // 2)
    return np.asarray(img.crop((cx, cy, cx + 224, cy + 224)))

@torch.no_grad()
def embed_batch(arrs):
    x = torch.from_numpy(np.stack(arrs)).to(device).permute(0, 3, 1, 2).float() / 255.0
    x = (x - MEAN) / STD
    cls = model(x).last_hidden_state[:, 0]
    return torch.nn.functional.normalize(cls, dim=-1).cpu().numpy()

In [ ]:
# ---- Embed all rows (checkpointed, resumable) ----
import time, urllib.request, concurrent.futures

new_q = old_q.copy() if old_q is not None else np.zeros((N, dim), dtype=np.int8)
done = np.zeros(N, dtype=bool)
failed = np.zeros(N, dtype=bool)
if os.path.exists(CHECKPOINT):
    ck = np.load(CHECKPOINT)
    new_q, done, failed = ck["new_q"], ck["done"], ck["failed"]
    print(f"resuming: {int(done.sum())}/{N} done, {int(failed.sum())} failed")

dl_dir = "/content/downloaded-images"; os.makedirs(dl_dir, exist_ok=True)

def load_row(i):
    e = meta[i]
    p = find_image(e["cardId"])
    if p is None and DOWNLOAD_MISSING and e.get("imageURL"):
        local = os.path.join(dl_dir, e["cardId"].replace("/", "_"))
        try:
            if not os.path.exists(local):
                req = urllib.request.Request(e["imageURL"], headers={"User-Agent": "TCGer-index-rebuild"})
                with urllib.request.urlopen(req, timeout=30) as r:
                    open(local, "wb").write(r.read())
            p = local
        except Exception:
            return i, None
    if p is None:
        return i, None
    try:
        return i, preprocess(Image.open(p).convert("RGB"))
    except Exception:
        return i, None

pending = [i for i in range(N) if not done[i]]
print(f"embedding {len(pending)} rows...")
t0, processed, last_ckpt = time.time(), 0, 0
with concurrent.futures.ThreadPoolExecutor(16) as pool:
    batch_idx, batch_arr = [], []
    def flush():
        global processed
        if not batch_idx: return
        vecs = embed_batch(batch_arr)
        for j, v in zip(batch_idx, vecs):
            new_q[j] = np.clip(np.round(v * SCALE), -127, 127).astype(np.int8)
            done[j] = True
        processed += len(batch_idx)
        batch_idx.clear(); batch_arr.clear()
    for i, arr in pool.map(load_row, pending):
        if arr is None:
            failed[i] = True; done[i] = True; processed += 1
        else:
            batch_idx.append(i); batch_arr.append(arr)
            if len(batch_idx) >= BATCH_SIZE: flush()
        if processed - last_ckpt >= 2000:
            last_ckpt = processed
            np.savez(CHECKPOINT, new_q=new_q, done=done, failed=failed)
            rate = processed / (time.time() - t0)
            print(f"{int(done.sum())}/{N} (failed {int(failed.sum())}) — {rate:.0f}/s, eta {(len(pending)-processed)/max(rate,1)/60:.0f} min")
    flush()
np.savez(CHECKPOINT, new_q=new_q, done=done, failed=failed)
print(f"done: {int(done.sum())}/{N} rows, {int(failed.sum())} kept fallback vectors")

In [ ]:
# ---- Write the bin + verify ----
with open(OUT_BIN, "wb") as f:
    f.write(struct.pack("<ii", N, dim))
    f.write(new_q.tobytes())
print(f"wrote {OUT_BIN} ({(8 + N*dim)/1e6:.1f} MB)")

if old_q is not None:
    a = new_q.astype(np.float32) / SCALE
    b = old_q.astype(np.float32) / SCALE
    a /= np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
    b /= np.maximum(np.linalg.norm(b, axis=1, keepdims=True), 1e-12)
    changed = ~failed & done
    cos = (a[changed] * b[changed]).sum(axis=1)
    print(f"new-vs-old row agreement on {int(changed.sum())} re-embedded rows: "
          f"mean {cos.mean():.3f}, p5 {np.percentile(cos, 5):.3f} "
          f"(expected ~0.95 mean — the q8 gap this rebuild removes)")
    # Self-retrieval sanity: every re-embedded row must be its own nearest neighbor.
    sample = np.flatnonzero(changed)[::max(1, int(changed.sum()) // 500)]
    top1 = (a[sample] @ a.T).argmax(axis=1)
    print(f"self-retrieval on {len(sample)} sampled rows: {(top1 == sample).mean()*100:.1f}% top-1")

## Ship it

1. Download `CardsIndexVectors-parity.bin` from the Drive folder.
2. Replace `mobile-apps/ios/TCGer/TCGer/Resources/ScanIndex/CardsIndexVectors.bin` with it (keep the original name).
3. Rebuild and reinstall the iOS app. `CardsIndexMetadata.json`, the gate, and the CoreML model are unchanged.
4. Do **not** replace the web index (`frontend/public/scan-index/pokemon-embeddings.json`) — the browser runs the q8 ONNX model, so its index must stay q8-built.
5. Delete `parity-rebuild-checkpoint.npz` from Drive once you're happy with the result.